# Climate Change Visualization — Future Precipitation & Temperature vs Historical Baseline

Reads monthly climatology for 4 climate models (Hot-Wet, Hot-Dry, Cool-Wet,
Cool-Dry) x 2 SSP scenarios (245, 585) x 3 future periods (Near, Mid, Far),
and expresses all future values as change relative to the historical
baseline — % change for precipitation (ratio-scale), absolute °C change for
Tmax/Tmin (interval-scale, where 0 isn't "none").

For each station x variable, produces 6 figures (annual change bars, monthly
change grid, heatmap, seasonal bars, boxplot spread, model-agreement chart)
via one shared, config-driven function

**Reads:** `New_Future_Climate/{ppt,tmax,tmin}/station_data_monthly/{station}_monthly_future.csv`
(from `2_future_climate_data.ipynb`).

**Writes:** figures and a long-format change CSV per station x variable, to
`New_Future_Climate/{variable}/station_plots/{station}/`

In [ ]:
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os


## Station lists

Derived from the historical monthly climatology files (same source
`2_future_climate_data.ipynb` and `3_future_climate_annual.ipynb` use) instead
of a hardcoded list, so all three notebooks can't silently drift apart on
which stations exist.

In [ ]:
historical_monthly_pcp = pd.read_csv('../All_DATA/Climate/Historical/Monthly/historical_monthly_average_pcp.csv')
historical_monthly_tmax = pd.read_csv('../All_DATA/Climate/Historical/Monthly/historical_monthly_average_tmax.csv')

pcp_stations = historical_monthly_pcp.columns[1:].tolist()
temp_stations = historical_monthly_tmax.columns[1:].tolist()

pcp_stations, temp_stations


## Config & per-station, per-variable figure generator

In [ ]:
# ------------------------------------------------------------------
# 0. CONFIG (shared across all variables)
# ------------------------------------------------------------------
MODEL_ORDER = ["Hot-Wet", "Hot-Dry", "Cool-Wet", "Cool-Dry"]
SCEN_ORDER = ["245", "585"]
PERIOD_ORDER = ["Near", "Mid", "Far"]
MONTH_NAMES = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
SEASON_MAP = {12: "DJF", 1: "DJF", 2: "DJF",
              3: "MAM", 4: "MAM", 5: "MAM",
              6: "JJA", 7: "JJA", 8: "JJA",
              9: "SON", 10: "SON", 11: "SON"}
SEASON_TITLES = {"DJF": "Winter (DJF)", "MAM": "Pre-monsoon (MAM)",
                  "JJA": "Monsoon (JJA)", "SON": "Post-monsoon (SON)"}

MODEL_COLORS = {
    "Hot-Wet":  "#1b7837",   # green
    "Hot-Dry":  "#d73027",   # red
    "Cool-Wet": "#4575b4",   # blue
    "Cool-Dry": "#fdae61",   # orange
}

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

# ------------------------------------------------------------------
# Per-variable config. This is the ONLY place that encodes what makes
# precipitation different from temperature:
#   - agg: "sum" for precip (monthly totals accumulate into an annual/
#          seasonal total), "mean" for temperature (does not accumulate)
#   - change: "pct" for precip (ratio-scale, meaningful % change),
#             "abs" for temperature (interval-scale, 0 isn't "none",
#             so absolute degrees change is used instead)
#   - direction_labels: labels for the fig6 agreement chart
# ------------------------------------------------------------------
VAR_CONFIG = {
    "ppt": {
        "folder": "ppt",
        "var_name": "Precipitation",
        "stations": [],          # fill in with pcp_stations
        "agg": "sum",
        "change": "pct",
        "unit": "%",
        "cmap": "RdBu",
        "direction_labels": ("Wetter", "Drier"),
        "direction_colors": ("#4575b4", "#d73027"),
    },
    "tmax": {
        "folder": "tmax",
        "var_name": "Maximum Temperature",
        "stations": [],          # fill in with temp_stations
        "agg": "mean",
        "change": "abs",
        "unit": "\u00b0C",
        "cmap": "RdBu_r",
        "direction_labels": ("Warmer", "Cooler"),
        "direction_colors": ("#d73027", "#4575b4"),
    },
    "tmin": {
        "folder": "tmin",
        "var_name": "Minimum Temperature",
        "stations": [],          # fill in with temp_stations
        "agg": "mean",
        "change": "abs",
        "unit": "\u00b0C",
        "cmap": "RdBu_r",
        "direction_labels": ("Warmer", "Cooler"),
        "direction_colors": ("#d73027", "#4575b4"),
    },
}


def process_station_variable(station, var_key):
    cfg = VAR_CONFIG[var_key]
    var_name = cfg["var_name"]
    unit = cfg["unit"]
    is_pct = cfg["change"] == "pct"
    agg_func = "sum" if cfg["agg"] == "sum" else "mean"
    warm_label, cool_label = cfg["direction_labels"]
    warm_color, cool_color = cfg["direction_colors"]

    DATA_PATH = f"../All_DATA/Climate/New_Future_Climate/{cfg['folder']}/station_data_monthly/{station}_monthly_future.csv"
    OUT_DIR = f"../All_DATA/Climate/New_Future_Climate/{cfg['folder']}/station_plots/{station}"
    os.makedirs(OUT_DIR, exist_ok=True)

    # ------------------------------------------------------------------
    # 1. LOAD + RESHAPE
    # ------------------------------------------------------------------
    df = pd.read_csv(DATA_PATH, sep=None, engine="python")  # auto-detect tab/comma
    df.columns = [c.strip() for c in df.columns]

    month_col = df.columns[0]
    name_to_num = {m: i + 1 for i, m in enumerate(MONTH_NAMES)}
    df["MonthNum"] = df[month_col].map(name_to_num)
    if df["MonthNum"].isna().any():
        raise ValueError(f"Could not map some values in '{month_col}' to months: "
                          f"{df.loc[df['MonthNum'].isna(), month_col].tolist()}")

    hist_col = [c for c in df.columns if "hist" in c.lower()][0]
    future_cols = [c for c in df.columns if c not in (month_col, hist_col, "MonthNum")]

    # Naming scheme: "<scenario>_<Model>_<period>", e.g. "245_Cool-Dry_near"
    pattern = re.compile(r"^(\d{3})_(Hot-Wet|Hot-Dry|Cool-Wet|Cool-Dry)_(near|mid|far)$", re.IGNORECASE)

    records = []
    for col in future_cols:
        m = pattern.match(col)
        if not m:
            print(f"WARNING [{station}/{var_key}]: could not parse column '{col}', skipping")
            continue
        scen, model, period = m.groups()
        period = period.capitalize()
        for _, row in df.iterrows():
            month = int(row["MonthNum"])
            hist_val = row[hist_col]
            fut_val = row[col]
            if is_pct:
                change = (fut_val - hist_val) / hist_val * 100 if hist_val != 0 else np.nan
            else:
                change = fut_val - hist_val
            records.append({
                "Month": month,
                "MonthName": MONTH_NAMES[month - 1],
                "Model": model,
                "Scenario": scen,
                "Period": period,
                "Historical": hist_val,
                "Future": fut_val,
                "Change": change,
            })

    long_df = pd.DataFrame(records)
    long_df["Model"] = pd.Categorical(long_df["Model"], MODEL_ORDER, ordered=True)
    long_df["Scenario"] = pd.Categorical(long_df["Scenario"], SCEN_ORDER, ordered=True)
    long_df["Period"] = pd.Categorical(long_df["Period"], PERIOD_ORDER, ordered=True)
    long_df.sort_values(["Model", "Scenario", "Period", "Month"], inplace=True)

    long_df.to_csv(os.path.join(OUT_DIR, f"change_long_format_station{station}_{var_key}.csv"), index=False)
    print(f"[{station}/{var_key}] Reshaped to long format: {long_df.shape[0]} rows")

    # ------------------------------------------------------------------
    # 2. ANNUAL AGGREGATE CHANGE (sum for precip, mean for temperature)
    # ------------------------------------------------------------------
    annual_hist_agg = df[hist_col].sum() if agg_func == "sum" else df[hist_col].mean()

    annual = (
        long_df.groupby(["Model", "Scenario", "Period"], observed=True)["Future"]
        .agg(agg_func)
        .reset_index()
        .rename(columns={"Future": "AnnualFuture"})
    )
    if is_pct:
        annual["ChangeAnnual"] = (annual["AnnualFuture"] - annual_hist_agg) / annual_hist_agg * 100
    else:
        annual["ChangeAnnual"] = annual["AnnualFuture"] - annual_hist_agg

    # ------------------------------------------------------------------
    # 3. SEASONAL AGGREGATION (meteorological seasons)
    # ------------------------------------------------------------------
    long_df["Season"] = long_df["Month"].map(SEASON_MAP)
    df["Season"] = df["MonthNum"].map(SEASON_MAP)

    hist_season_agg = df.groupby("Season")[hist_col].agg(agg_func)

    seasonal = (
        long_df.groupby(["Model", "Scenario", "Period", "Season"], observed=True)["Future"]
        .agg(agg_func)
        .reset_index()
        .rename(columns={"Future": "SeasonFuture"})
    )
    seasonal["HistAgg"] = seasonal["Season"].map(hist_season_agg)
    if is_pct:
        seasonal["ChangeSeason"] = (seasonal["SeasonFuture"] - seasonal["HistAgg"]) / seasonal["HistAgg"] * 100
    else:
        seasonal["ChangeSeason"] = seasonal["SeasonFuture"] - seasonal["HistAgg"]
    seasonal["Season"] = pd.Categorical(seasonal["Season"], ["DJF", "MAM", "JJA", "SON"], ordered=True)

    def pct_or_unit_label(base):
        return f"{base} (%)" if is_pct else f"{base} ({unit})"

    # ==================================================================
    # FIGURE 1 — Annual change: grouped bar chart
    # ==================================================================
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharey=True)
    bar_width = 0.2

    for ax, scen in zip(axes, SCEN_ORDER):
        sub = annual[annual["Scenario"] == scen]
        x = np.arange(len(PERIOD_ORDER))
        for i, model in enumerate(MODEL_ORDER):
            vals = [sub[(sub["Model"] == model) & (sub["Period"] == p)]["ChangeAnnual"].values[0]
                    for p in PERIOD_ORDER]
            ax.bar(x + (i - 1.5) * bar_width, vals, width=bar_width,
                   label=model, color=MODEL_COLORS[model])
        ax.axhline(0, color="black", linewidth=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(PERIOD_ORDER)
        ax.set_title(f"SSP{scen}")
        ax.set_xlabel("Future Period")
        if is_pct:
            ax.yaxis.set_major_formatter(mticker.PercentFormatter())

    agg_word = "Total" if agg_func == "sum" else "Mean"
    axes[0].set_ylabel(pct_or_unit_label(f"Change in Annual {agg_word} {var_name} vs Historical"))
    axes[0].legend(title="Model", loc="upper left", fontsize=8)
    fig.suptitle(f"Projected Change in Annual {agg_word} {var_name}_St{station}", fontsize=13, y=1.02)
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, f"fig1_annual_change_bars_station{station}_{var_key}.png"), bbox_inches="tight")
    plt.close(fig)

    # ==================================================================
    # FIGURE 2 — Monthly change line plots, faceted by Scenario x Period
    # ==================================================================
    fig, axes = plt.subplots(2, 3, figsize=(16, 8.5), sharey=True, sharex=True)
    for i, scen in enumerate(SCEN_ORDER):
        for j, period in enumerate(PERIOD_ORDER):
            ax = axes[i, j]
            sub = long_df[(long_df["Scenario"] == scen) & (long_df["Period"] == period)]
            for model in MODEL_ORDER:
                m_sub = sub[sub["Model"] == model].sort_values("Month")
                ax.plot(m_sub["Month"], m_sub["Change"],
                        color=MODEL_COLORS[model], label=model, marker="o", markersize=3, linewidth=1.8)
            ax.axhline(0, color="black", linewidth=0.8, linestyle=":")
            if is_pct:
                # Symlog keeps small % changes readable while still showing very
                # large spikes that occur when the historical baseline is near zero.
                ax.set_yscale("symlog", linthresh=100, linscale=0.5)
                ax.yaxis.set_major_formatter(mticker.PercentFormatter())
            ax.set_title(f"SSP{scen} \u2014 {period} Future", fontsize=10)
            ax.set_xticks(range(1, 13))
            ax.set_xticklabels(MONTH_NAMES, rotation=45, fontsize=7)
        axes[i, 0].set_ylabel(pct_or_unit_label("Change vs Historical") + ("\n(symlog scale)" if is_pct else ""))

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, title="Model", loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.06))
    subtitle = "\n(note: symlog y-axis \u2014 huge % spikes occur when historical baseline is near zero)" if is_pct else ""
    fig.suptitle(f"Monthly {var_name} Change vs Historical_St{station}{subtitle}", fontsize=12, y=1.13 if is_pct else 1.1)
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, f"fig2_monthly_change_grid_station{station}_{var_key}.png"), bbox_inches="tight")
    plt.close(fig)

    # ==================================================================
    # FIGURE 3 — Heatmap: rows = Model-Scenario, cols = Month, per Period
    # ==================================================================
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
    vmax = np.nanmax(np.abs(long_df["Change"]))
    if is_pct:
        vmax = min(vmax, 300)  # clip extreme outliers for a readable color scale

    row_labels = [f"{m} SSP{s}" for m in MODEL_ORDER for s in SCEN_ORDER]

    for ax, period in zip(axes, PERIOD_ORDER):
        sub = long_df[long_df["Period"] == period]
        matrix = np.zeros((len(row_labels), 12))
        for r, lbl in enumerate(row_labels):
            model, scen = lbl.split(" SSP")
            row = sub[(sub["Model"] == model) & (sub["Scenario"] == scen)].sort_values("Month")
            matrix[r, :] = row["Change"].values
        im = ax.imshow(matrix, cmap=cfg["cmap"], vmin=-vmax, vmax=vmax, aspect="auto")
        ax.set_xticks(range(12))
        ax.set_xticklabels(MONTH_NAMES, rotation=45, fontsize=8)
        ax.set_yticks(range(len(row_labels)))
        ax.set_yticklabels(row_labels, fontsize=8)
        ax.set_title(f"{period} Future", fontsize=11)

    cbar = fig.colorbar(im, ax=axes, orientation="vertical", fraction=0.02, pad=0.02)
    cbar.set_label(pct_or_unit_label(f"Change in {var_name} vs Historical"))
    fig.suptitle(f"Heatmap of Monthly Change by Model_St{station} \u00d7 Scenario \u00d7 Period ({var_key})", fontsize=13, y=1.03)
    fig.savefig(os.path.join(OUT_DIR, f"fig3_heatmap_month_model_scenario_station{station}_{var_key}.png"), bbox_inches="tight")
    plt.close(fig)

    # ==================================================================
    # FIGURE 4 — Seasonal change grouped bars (facet per Season)
    # ==================================================================
    seasons = ["DJF", "MAM", "JJA", "SON"]

    fig, axes = plt.subplots(2, 4, figsize=(18, 8.5), sharey=True, sharex=True)
    x = np.arange(len(PERIOD_ORDER))
    bar_width = 0.2

    for i, scen in enumerate(SCEN_ORDER):
        for j, season in enumerate(seasons):
            ax = axes[i, j]
            sub = seasonal[(seasonal["Season"] == season) & (seasonal["Scenario"] == scen)]
            for k, model in enumerate(MODEL_ORDER):
                vals = []
                for p in PERIOD_ORDER:
                    row = sub[(sub["Model"] == model) & (sub["Period"] == p)]
                    vals.append(row["ChangeSeason"].values[0] if not row.empty else np.nan)
                ax.bar(x + (k - 1.5) * bar_width, vals, width=bar_width,
                       color=MODEL_COLORS[model], label=model if (i == 0 and j == 0) else None)
            ax.axhline(0, color="black", linewidth=0.8)
            ax.set_xticks(x)
            ax.set_xticklabels(PERIOD_ORDER)
            if is_pct:
                ax.yaxis.set_major_formatter(mticker.PercentFormatter())
            if i == 0:
                ax.set_title(SEASON_TITLES[season], fontsize=11)
        axes[i, 0].set_ylabel(f"SSP{scen}\n" + pct_or_unit_label("Change vs Historical"))

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, title="Model", loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.06))
    fig.suptitle(f"Seasonal {var_name} Change vs Historical_St{station}", fontsize=13, y=1.12)
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, f"fig4_seasonal_change_station{station}_{var_key}.png"), bbox_inches="tight")
    plt.close(fig)

    # ==================================================================
    # FIGURE 5 — Distribution (boxplot) of monthly change by Period, faceted by Scenario
    # ==================================================================
    fig, axes = plt.subplots(1, 2, figsize=(11, 5.5), sharey=True)
    for ax, scen in zip(axes, SCEN_ORDER):
        data_to_plot = [long_df[(long_df["Scenario"] == scen) & (long_df["Period"] == p)]["Change"].values
                        for p in PERIOD_ORDER]
        bp = ax.boxplot(data_to_plot, patch_artist=True, showmeans=True)
        ax.set_xticks(np.arange(1, len(PERIOD_ORDER) + 1))
        ax.set_xticklabels(PERIOD_ORDER)
        for patch, color in zip(bp["boxes"], ["#a6d96a", "#fdae61", "#d73027"]):
            patch.set_facecolor(color)
            patch.set_alpha(0.6)
        ax.axhline(0, color="black", linewidth=0.8, linestyle=":")
        ax.set_title(f"SSP{scen}")
        ax.set_xlabel("Future Period")
        if is_pct:
            ax.yaxis.set_major_formatter(mticker.PercentFormatter())

    axes[0].set_ylabel(pct_or_unit_label("Change vs Historical") + "\n(spread across all months & models)")
    fig.suptitle(f"Uncertainty/Spread of {var_name} Change by Period_St{station}", fontsize=13, y=1.02)
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, f"fig5_boxplot_spread_by_period_station{station}_{var_key}.png"), bbox_inches="tight")
    plt.close(fig)

    # ==================================================================
    # FIGURE 6 — Model agreement on direction of change
    # (Wetter/Drier for precip, Warmer/Cooler for temperature)
    # ==================================================================
    fig, axes = plt.subplots(1, 2, figsize=(11, 5.5), sharey=True)
    for ax, scen in zip(axes, SCEN_ORDER):
        up_pct, down_pct = [], []
        for period in PERIOD_ORDER:
            sub = long_df[(long_df["Scenario"] == scen) & (long_df["Period"] == period)]
            up_pct.append((sub["Change"] > 0).mean() * 100)
            down_pct.append((sub["Change"] <= 0).mean() * 100)
        x = np.arange(len(PERIOD_ORDER))
        ax.bar(x, up_pct, color=warm_color, label=f"{warm_label} (months\u00d7models)")
        ax.bar(x, [-d for d in down_pct], color=cool_color, label=f"{cool_label} (months\u00d7models)")
        ax.axhline(0, color="black", linewidth=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(PERIOD_ORDER)
        ax.set_title(f"SSP{scen}")
        ax.set_ylim(-100, 100)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{abs(v):.0f}%"))

    axes[0].set_ylabel("Share of Model-Month Combinations")
    axes[0].legend(loc="lower left", fontsize=8)
    fig.suptitle(f"Direction of Change Agreement: {warm_label} vs {cool_label} Months Across Models_St{station} ({var_key})",
                 fontsize=13, y=1.02)
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, f"fig6_direction_agreement_station{station}_{var_key}.png"), bbox_inches="tight")
    plt.close(fig)

    print(f"[{station}/{var_key}] All figures saved to:", os.path.abspath(OUT_DIR))
    print(f"[{station}/{var_key}] Annual change summary:")
    print(annual.pivot_table(index=["Model"], columns=["Scenario", "Period"], values="ChangeAnnual").round(2))
    print()

## Run for every station x variable

In [ ]:
# ------------------------------------------------------------------
# MAIN: loop over every variable and every station configured for it
# ------------------------------------------------------------------
if __name__ == "__main__":
    # Point each variable at its station list before running, e.g.:
    VAR_CONFIG["ppt"]["stations"] = pcp_stations
    VAR_CONFIG["tmax"]["stations"] = temp_stations
    VAR_CONFIG["tmin"]["stations"] = temp_stations

    for var_key, cfg in VAR_CONFIG.items():
        for station in cfg["stations"]:
            process_station_variable(station, var_key)